In [ ]:
from solver.optimizer import optimize_camera_trajectory

In [ ]:
import math
def make_subject_tracks(
        total_frames=1200, image_w=1920, image_h=1080):
    tracks = {}
    A = []
    for f in range(total_frames):
        u = f / (total_frames - 1)
        cx = -2.0 + 4.0*u
        cz =  0.5 + 1.0*u
        cy =  1.6  
        h = 320 + int(80*math.sin(2*math.pi*u))
        w = int(h * 0.45)
        px = int(image_w*0.5 + 250*(cx/4.0))
        py = int(image_h*0.55 - 60*(cz/2.0))
        bbox = {"x1":1, "y1": 1, "x2": 1, "y2":1}
        A.append({"bbox": bbox, "C0": [0 , 0 , 0]})
    tracks["C0"] = A
    B = []
    for f in range(total_frames):
        u = f / (total_frames - 1)
        cx = 1.0
        cz = 2.5
        cy = 1.6 + 0.05*math.sin(4*math.pi*u)
        h = 220
        w = int(h * 0.45)
        px = int(image_w*0.65)
        py = int(image_h*0.58)
        bbox = {"x1": px-w//2, "y1": py-h, "x2": px+w//2, "y2": py}
        B.append({"bbox": bbox, "C0": [cx, cy, cz]})
    tracks["B"] = B

    return tracks

In [ ]:
start_pose = {
    "kind": "point",
    "t": 0,
    "position": [0.0, 1.6, -6.0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "losses": []
}

In [ ]:
def build_subject_centers(subject_tracks, total_frames, device):
    centers = {}

    for sid, track in subject_tracks.items():
        c = []
        for f in range(total_frames):
            c.append(track[f]["C0"])
        centers[sid] = torch.tensor(c, dtype=torch.float32, device=device)

    return centers

In [ ]:
subject_tracks = make_subject_tracks()
subject_centers = build_subject_centers(subject_tracks , 1200 , "cpu")

In [ ]:

start_pose = {
    "kind": "point",
    "t": 0,
    "position": [0.0, 1.6, -6.0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "losses": []
}
subject_centers = {
    "C0": torch.tensor([[0.0, 1.6, 0.0]] * 120, dtype=torch.float32, device="cpu")
}
constraints = [
    start_pose,
    
    {
        "kind": "interval",
        "t0": 0, "t1":  1200,
        "losses": [
            {"type": "truckRightMovement", "distance"  :3}
        ]
    }
]


out = optimize_camera_trajectory(constraints, duration_seconds=40, trajectory_mode="interpolation", spline_degree=3, subject_tracks=subject_tracks, subject_centers=subject_centers)

1200
15


c:\Users\mrami\anaconda3\envs\instantdrag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'minPath/interval': tensor(0.0250, dtype=torch.float64, grad_fn=<AddBackward0>), 'truckMovement/target_end': tensor(4.4100e-16, dtype=torch.float64, grad_fn=<AddBackward0>), 'truckMovement/monotonic': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'truckMovement/drift': tensor(4.0552e-30, dtype=torch.float64, grad_fn=<AddBackward0>), 'truckMovement/keepRot': tensor(4.0000e-08, dtype=torch.float64, grad_fn=<AddBackward0>)}
Iter    1 | Loss 0.025041 | mode=interpolation


In [11]:
P = out["P"]
Q = out["Q"]

In [10]:

start_pose = {
    "kind": "point",
    "t": 0,
    "position": [4, 0, 0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "losses": []
}
subject_centers = {
    "C0": torch.tensor([[0.0, 0, 0.0]] * 1200, dtype=torch.float32, device="cpu")
}
constraints = [
    start_pose,
    
    {
        "kind": "interval",
        "t0": 0, "t1":  1200,
        "k": 15,
        "losses": [
            {"type": "arcMovement","subjectId": "C0", "angleDeg": 360.0 , "radius" : 4}
        ]
    }
    # {
    #     "kind": "interval",
    #     "t0": 0, "t1":  1200,
    #     "k": 15,
    #     "losses": [
    #         {"type": "staticMovement"}
    #     ]
    # }
]

out = optimize_camera_trajectory(constraints, duration_seconds=40, trajectory_mode="matrix", spline_degree=12, subject_tracks=subject_tracks, subject_centers=subject_centers , loss_threshold=1)

1200
13
{'minPath/interval': tensor(0.1607, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane_step': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/radius_target': tensor(1408.3426, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_dir': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_target': tensor(9.8691e-15, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_uniform': tensor(0.0361, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_progress_spec': tensor(0.0188, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_step_cap': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/lookat': tensor(1.3989, dtype=torch.float64, grad_fn=<AddBackward0>)}
Iter    1 | Loss 1409.957037 | mode=matrix
{'minPath/interval': tensor(0.1611, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(3.7446e-05, dtype=torch.float64, grad_fn=<AddBackwar

In [ ]:
import torch
import matplotlib.pyplot as plt

def _to_torch(a, device=None, dtype=torch.float64):
    if a is None:
        return None
    if torch.is_tensor(a):
        t = a
    else:
        t = torch.as_tensor(a)
    if dtype is not None:
        t = t.to(dtype=dtype)
    if device is not None:
        t = t.to(device)
    return t

def _q_normalize_t(q, eps=1e-8):
    q = _to_torch(q, dtype=torch.float64)
    n = torch.linalg.norm(q, dim=-1, keepdim=True)
    return q / (n + eps)

def _q_conj_t(q):
    q = _to_torch(q, dtype=torch.float64)
    out = q.clone()
    out[..., 1:] = -out[..., 1:]
    return out

def _q_mul_t(a, b):
    a = _to_torch(a, dtype=torch.float64)
    b = _to_torch(b, dtype=torch.float64)

    aw, ax, ay, az = a.unbind(dim=-1)
    bw, bx, by, bz = b.unbind(dim=-1)

    w = aw*bw - ax*bx - ay*by - az*bz
    x = aw*bx + ax*bw + ay*bz - az*by
    y = aw*by - ax*bz + ay*bw + az*bx
    z = aw*bz + ax*by - ay*bx + az*bw
    return torch.stack([w, x, y, z], dim=-1)

def _q_rotate_t(q, v):
    q = _q_normalize_t(q)

    v = _to_torch(v, dtype=torch.float64)
    if v.ndim == 1:
        if q.ndim == 2:
            v = v.unsqueeze(0).expand(q.shape[0], -1)

    zeros = torch.zeros(v.shape[:-1] + (1,), dtype=v.dtype, device=v.device)
    vq = torch.cat([zeros, v], dim=-1)

    return _q_mul_t(_q_mul_t(q, vq), _q_conj_t(q))[..., 1:]

def plot_points_trajectory(
    P,
    Q=None,
    fps=30,
    ax=None,
    title="Camera trajectory",
    arrow_every=None,
    arrow_len=None,
    forward_local=(0, 0, 1), 
):

    P = _to_torch(P, dtype=torch.float64)
    Q = _to_torch(Q, dtype=torch.float64) if Q is not None else None

    if P.ndim != 2 or P.shape[1] != 3:
        raise ValueError(f"P must have shape (N,3), got {tuple(P.shape)}")

    N = P.shape[0]
    if N == 0:
        raise ValueError("P is empty")

    if Q is not None:
        if Q.ndim != 2 or Q.shape[1] != 4:
            raise ValueError(f"Q must have shape (N,4), got {tuple(Q.shape)}")
        if Q.shape[0] != N:
            raise ValueError(f"P and Q must have same length, got {N} and {Q.shape[0]}")

    Pc = P.detach().cpu()
    x, y, z = Pc[:, 0], Pc[:, 1], Pc[:, 2]
    Xp = x
    Yp = z
    Zp = y

    created_ax = False
    if ax is None:
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection="3d")
        created_ax = True

    ax.plot(Xp, Yp, Zp, marker="o", markersize=3, linewidth=1)

    ax.scatter([Xp[0].item()],  [Yp[0].item()],  [Zp[0].item()],  s=140, marker="*", label="Start")
    ax.scatter([Xp[-1].item()], [Yp[-1].item()], [Zp[-1].item()], s=120, marker="X", label="End")

    t0_sec = 0.0
    t1_sec = (N - 1) / float(fps)
    ax.text(Xp[0].item(),  Yp[0].item(),  Zp[0].item(),  f"  START\n  f=0, t={t0_sec:.2f}s")
    ax.text(Xp[-1].item(), Yp[-1].item(), Zp[-1].item(), f"  END\n  f={N-1}, t={t1_sec:.2f}s")

    if Q is not None:
        if arrow_every is None:
            arrow_every = max(1, N // 20)

        idx = torch.arange(0, N, step=arrow_every, device=Q.device, dtype=torch.long)
        if idx.numel() == 0:
            idx = torch.tensor([0], device=Q.device, dtype=torch.long)
        if idx[-1].item() != N - 1:
            idx = torch.cat([idx, torch.tensor([N - 1], device=Q.device, dtype=torch.long)], dim=0)

        Qs = _q_normalize_t(Q[idx])

        fwd_local = _to_torch(forward_local, device=Qs.device, dtype=torch.float64)
        fwd = _q_rotate_t(Qs, fwd_local)

        fwd_norm = torch.linalg.norm(fwd, dim=-1, keepdim=True)
        fwd = fwd / (fwd_norm + 1e-8)

        if arrow_len is None:
            span = (Pc.max(dim=0).values - Pc.min(dim=0).values)
            diag = torch.linalg.norm(span)
            arrow_len = float(0.06 * diag) if float(diag) > 1e-8 else 0.2

        d_world = fwd * float(arrow_len)

        dXp = d_world[:, 0].detach().cpu()
        dYp = d_world[:, 2].detach().cpu()
        dZp = d_world[:, 1].detach().cpu()

        PX = Xp[idx.detach().cpu()]
        PY = Yp[idx.detach().cpu()]
        PZ = Zp[idx.detach().cpu()]

        ax.quiver(
            PX, PY, PZ,
            dXp, dYp, dZp,
            length=1.0,
            normalize=False,
            arrow_length_ratio=0.25
        )

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("z")
    ax.set_zlabel("y (vertical)")
    ax.legend()

    xr = float(Xp.max() - Xp.min())
    yr = float(Yp.max() - Yp.min())
    zr = float(Zp.max() - Zp.min())
    xr = xr if xr > 1e-6 else 1.0
    yr = yr if yr > 1e-6 else 1.0
    zr = zr if zr > 1e-6 else 1.0
    try:
        ax.set_box_aspect((xr, yr, zr))
    except Exception:
        pass

    if created_ax:
        plt.show()

    return ax

In [26]:

start_pose = {
    "kind": "point",
    "t": 0,
    "position": [0.0, 0, 0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "losses": []
}
subject_centers = {
    "C0": torch.tensor([[0.0, 1.6, 0.0]] * 1200, dtype=torch.float32, device="cpu")
}
constraints = [
    start_pose,
    
    {
        "kind": "interval",
        "t0": 0, "t1":  1200,
        "k": 15,
        "losses": [
            {"type": "arcMovement","subjectId": "C0", "angleDeg": 540.0 , "radius" : 4}
        ]
    }
]

out = optimize_camera_trajectory(constraints, duration_seconds=40, trajectory_mode="matrix", spline_degree=9, subject_tracks=subject_tracks, subject_centers=subject_centers , loss_threshold=1)
P = out["P"]
Q = out["Q"]

1200
19
{'minPath/interval': tensor(0.2748, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane_step': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/radius_target': tensor(960.7750, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_dir': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_target': tensor(0.6738, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_uniform': tensor(0.4247, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_progress_spec': tensor(0.3683, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_step_cap': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/lookat': tensor(42.6401, dtype=torch.float64, grad_fn=<AddBackward0>)}
Iter    1 | Loss 1005.156694 | mode=matrix
{'minPath/interval': tensor(0.2750, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/p

In [27]:
P = out["P"]
Q = out["Q"]

In [30]:

start_pose = {
    "kind": "point",
    "t": 0,
    "position": [0.0, 0, 0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
    "losses": []
}
subject_centers = {
    "C0": torch.tensor([[0.0, 1.6, 0.0]] * 1200, dtype=torch.float32, device="cpu")
}
constraints = [
    start_pose,
    
    {
        "kind": "interval",
        "t0": 0, "t1":  1200,
        "k": 15,
        "losses": [
            {"type": "arcMovement","subjectId": "C0", "angleDeg": 720.0 , "radius" : 4}
        ]
    }
]

out = optimize_camera_trajectory(constraints, duration_seconds=40, trajectory_mode="matrix", spline_degree=9, subject_tracks=subject_tracks, subject_centers=subject_centers , loss_threshold=1)
P = out["P"]
Q = out["Q"]

1200
25
{'minPath/interval': tensor(0.3683, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane_step': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/radius_target': tensor(886.0199, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_dir': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_target': tensor(3.9439e-14, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_uniform': tensor(0.4012, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_progress_spec': tensor(0.3541, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/angle_step_cap': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/lookat': tensor(46.1200, dtype=torch.float64, grad_fn=<AddBackward0>)}
Iter    1 | Loss 933.263570 | mode=matrix
{'minPath/interval': tensor(0.3685, dtype=torch.float64, grad_fn=<AddBackward0>), 'arc/plane': tensor(0., dtype=torch.float64, grad_fn=<AddBackward0>), 'ar